# Day 5 — Delta: History, Time Travel, Optimize & Ops

Allow about 1.5–2 hours. Sections on CDF (6b), CHECK (7b), and the recap quiz can be done later if you run short on time.

Use notebook 01 first so `P_BASIC` and `P_PART` exist.

You will inspect history, use version and timestamp time travel, call `DeltaTable.history`, read about RESTORE and shallow clone (commented patterns), run OPTIMIZE and optional ZORDER, review VACUUM and CDF concepts, try optional hands-on cells for CDF and CHECK where your runtime allows, and work through extra drills and a short quiz.

Standard Delta versioning and optimization labs from Databricks training map well to this content.

## Setup

In [0]:
%run ./02-Mount-Azure-Data-Lake

# Azure Data Lake Storage Gen2 — ABFS + OAuth (no DBFS mount)

This notebook configures **Spark** to read/write Azure Data Lake Storage Gen2 using the **ABFS** URI scheme (`abfss://...`) and **OAuth** via `spark.conf` (Service Principal). **We do not use `dbutils.fs.mount`** here.

---

## Prerequisites

1. ADLS Gen2 storage account and container with your data.
2. Service Principal with **Storage Blob Data Contributor** (or equivalent) on the container.
3. Fill in **tenant_id**, **client_id**, **client_secret** below (or use `dbutils.secrets.get` for production).

**Expected data layout** under the container (folder `data/`):
- `data/json/2015-summary.json`
- `data/2010-summary.csv`

After this runs, `base_path` points at `abfss://<container>@<account>.dfs.core.windows.net/data`.

---

## How lesson notebooks connect

**All hands-on notebooks that read ADLS** (`01-Day1-...`, `01-Day2-...`, `01-Day3-...`, `03-Day3-...`, `01-Day4-...`, `03-Day4-...`, `01-Day5-...`, `03-Day5-...`, `04-Day5-...`) must start with **`%run ./02-Mount-Azure-Data-Lake`** (typically the **first code cell**) so **`spark.conf` OAuth** and **`base_path`** are set on the cluster.

- Keep **`02-Mount-Azure-Data-Lake.ipynb`** in the **same workspace folder** as the lesson notebook, or adjust the path (e.g. `%run ../notebooks/02-Mount-Azure-Data-Lake`).
- After a **new cluster** or **RESTART**, run **`%run`** again, then the lesson **paths** cell (`BASE_PATH = base_path`, etc.).
- You can **Run all** on this notebook alone for debugging; in class it is normally executed **only** via `%run` from other notebooks.

## Step 1 — Storage account and OAuth (Service Principal)

OAuth configured for account: sadbtrng19032026


## Step 2 — Base path (ABFSS) and optional smoke read

base_path: abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data


+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
|Equatorial Guinea|      United States|    1|
+-----------------+-------------------+-----+
only showing top 5 rows


In [0]:
BASE_PATH = base_path
STUDENT_ID = "u01"  # Change to your assigned u01–u16
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
P_PART = DAY5 + "/delta/flight_partitioned"

# Verify prerequisite tables from Notebook 01 exist
print("=== Checking Day 5 Notebook 01 Prerequisites ===")
prereqs_met = True
try:
    spark.read.format("delta").load(P_BASIC).limit(1).collect()
    print(f"✓ P_BASIC table exists: {P_BASIC}")
except Exception as e:
    print(f"✗ P_BASIC table NOT found: {P_BASIC}")
    print(f"  Error: {type(e).__name__}")
    print(f"  Action: Run notebook 01 first to create this table")
    prereqs_met = False

try:
    spark.read.format("delta").load(P_PART).limit(1).collect()
    print(f"✓ P_PART table exists: {P_PART}")
except Exception as e:
    print(f"✗ P_PART table NOT found: {P_PART}")
    print(f"  Error: {type(e).__name__}")
    print(f"  Action: Run notebook 01 first to create this table")
    prereqs_met = False

if not prereqs_met:
    print("\n⚠️  WARNING: Day 5 Notebook 01 prerequisite tables are missing.")
    print("   Complete notebook 01 before running this notebook.")
else:
    print("\n✓ All prerequisites met for Day 5 Notebook 03")

print("\nUsing:", P_BASIC)

Using: abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/day5/delta/flight_summary_basic


## Section 1 — Deep **history** inspection

In [0]:
spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`").show(50, truncate=False)

+-------+-------------------+---------------+---------------------------------+---------+------------------------------------------------------------+----+------------------+--------------------+-----------+-----------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------+------------+------------------------------------------+
|version|timestamp          |userId         |userName                         |operation|operationParameters                                         |job |notebook          |clusterId           |readVersion|isolationLevel   |isBlindAppend|operationMetrics                                                                                                                         |userMetadata|engineInfo                                |
+-------+-------------------+---------------+---------------------------------+---------+-------------------------------------------

In [0]:
# Latest version number
from pyspark.sql.functions import col as C

h = spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`")
ver_max = h.selectExpr("max(version) as v").collect()[0]["v"]
ver_min = h.selectExpr("min(version) as v").collect()[0]["v"]
print("version min/max:", ver_min, ver_max)

version min/max: 0 1


## Section 2 — **Time travel** reads

In [0]:
from pyspark.sql.functions import col

v = int(spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`").selectExpr("max(version) as v").collect()[0]["v"])
if v >= 1:
    print("sum(count) at version v-1:")
    spark.read.format("delta").option("versionAsOf", v - 1).load(P_BASIC).groupBy().sum("count").show()
print("sum(count) at latest version", v)
spark.read.format("delta").option("versionAsOf", v).load(P_BASIC).groupBy().sum("count").show()

sum(count) at version v-1:
+----------+
|sum(count)|
+----------+
|    422269|
+----------+

sum(count) at latest version 1
+----------+
|sum(count)|
+----------+
|    844538|
+----------+



In [0]:
from pyspark.sql.functions import col as C

row = spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`").filter(C("version") == 0).select("timestamp").collect()
if row:
    ts = str(row[0]["timestamp"])
    c_ts = spark.read.format("delta").option("timestampAsOf", ts).load(P_BASIC).count()
    print("Row count TIMESTAMP AS OF version-0 time:", c_ts)
else:
    print("No history row for version 0")

Row count TIMESTAMP AS OF version-0 time: 255


## Section 2b — **Subtract** rows: latest vs previous version

In [0]:
from pyspark.sql.functions import col

h = spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`")
v = int(h.selectExpr("max(version) as v").collect()[0]["v"])
cols = ["DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count"]
cur = spark.read.format("delta").option("versionAsOf", v).load(P_BASIC).select(*cols)
if v >= 1:
    prev = spark.read.format("delta").option("versionAsOf", v - 1).load(P_BASIC).select(*cols)
    print("counts prev, cur:", prev.count(), cur.count())
    print("in cur but not prev:", cur.subtract(prev).count())
    print("in prev but not cur:", prev.subtract(cur).count())
else:
    print("Only one version; run notebook 01 append cells first.")

counts prev, cur: 255 510
in cur but not prev: 0
in prev but not cur: 0


## Section 2c — **`DeltaTable` API** for history (same log as SQL)

In [0]:
from delta.tables import DeltaTable

DeltaTable.forPath(spark, P_BASIC).history(15).select("version", "timestamp", "operation").show(15, truncate=False)

+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|1      |2026-03-21 13:53:36|WRITE    |
|0      |2026-03-21 13:53:22|WRITE    |
+-------+-------------------+---------+



## Section 3 — **RESTORE** (Databricks SQL)

Restore table state to an older version **creates a new version** that matches old data.

```sql
-- RESTORE TABLE delta.`abfss://.../path` TO VERSION AS OF 0;
```

Do not run RESTORE on shared class tables unless your workspace policy allows it; use a personal scratch path if you experiment.

## Section 3b — **SHALLOW CLONE** (pattern; optional)

Cheap **metadata** copy pointing at same files — great for experimentation / sharing snapshots.

```sql
-- CREATE TABLE delta.`abfss://.../day5/delta/clone_basic` SHALLOW CLONE delta.`abfss://.../day5/delta/flight_summary_basic`;
```

Requires write permission on the clone path. In shared training workspaces, use a path under your own `day5` prefix rather than a common table.

## Section 4 — **OPTIMIZE** + **ZORDER** (Databricks)

In [0]:
# File metrics before OPTIMIZE
spark.sql(f"DESCRIBE DETAIL delta.`{P_PART}`").select("numFiles", "sizeInBytes").show(truncate=False)

+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|22      |35302      |
+--------+-----------+



In [0]:
# Compacts small files (Databricks Delta)
spark.sql(f"OPTIMIZE delta.`{P_PART}`")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
# Z-order by high-cardinality filter columns (example)
try:
    spark.sql(f"OPTIMIZE delta.`{P_PART}` ZORDER BY (DEST_COUNTRY_NAME)")
except Exception as e:
    print("ZORDER may require Photon/Delta version:", type(e).__name__, e)

In [0]:
spark.sql(f"DESCRIBE DETAIL delta.`{P_PART}`").select("numFiles", "sizeInBytes").show(truncate=False)

+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|22      |35302      |
+--------+-----------+



## Section 4b — **OPTIMIZE** on **`P_BASIC`** (compare detail)

In [0]:
spark.sql(f"DESCRIBE DETAIL delta.`{P_BASIC}`").select("numFiles", "sizeInBytes").show(truncate=False)
spark.sql(f"OPTIMIZE delta.`{P_BASIC}`")
spark.sql(f"DESCRIBE DETAIL delta.`{P_BASIC}`").select("numFiles", "sizeInBytes").show(truncate=False)

+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|2       |13062      |
+--------+-----------+

+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|1       |7141       |
+--------+-----------+



## Section 4c — **Auto Optimize** (workspace / table settings — theory)

Many workspaces enable **automatic** `OPTIMIZE` / compaction policies on **managed** tables. For **external** `abfss://` paths (this course), you often **schedule** `OPTIMIZE` in **Jobs** after loads.

Discuss: **file size targets**, **cost vs query latency**, **concurrent writers**.

## Section 5 — **VACUUM** (discussion only)

`VACUUM` deletes data files **no longer referenced** by the table. Default **7-day** retention for time travel.

```python
# spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
# spark.sql(f"VACUUM delta.`{P_BASIC}` RETAIN 168 HOURS")
```

**Do not** run aggressive vacuum in class without admin sign-off.

## Section 6 — **Change Data Feed (CDF)** overview

CDF records row-level changes (`insert`, `update_preimage`, `update_postimage`, `delete`) for incremental downstream consumers.

Enable at table properties (example):

```sql
-- ALTER TABLE delta.`path` SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
```

Then `table_changes('path')` or Spark `readStream.format('delta').option('readChangeFeed','true')` — detailed in Day 6 / streaming modules.

## Section 6b — **Change Data Feed** — hands-on (optional; try/except)

In [0]:
from pyspark.sql.functions import lit

P_CDF = DAY5 + "/delta/cdf_lab_demo"
base = spark.read.format("delta").load(P_BASIC).select("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count").limit(30)
base.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P_CDF)
try:
    spark.sql(f"ALTER TABLE delta.`{P_CDF}` SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    print("CDF enabled on", P_CDF)
except Exception as e:
    print("ALTER TBLPROPERTIES:", type(e).__name__, str(e)[:200])

CDF enabled on abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/day5/delta/cdf_lab_demo


In [0]:
# Append rows to generate INSERT change records (after CDF enabled)
from pyspark.sql.functions import col, lit

try:
    extra = spark.read.format("delta").load(P_CDF).limit(3).withColumn("count", col("count") + lit(1))
    extra.write.format("delta").mode("append").save(P_CDF)
    print("Append OK — new history version should exist")
    spark.sql(f"DESCRIBE HISTORY delta.`{P_CDF}`").select("version", "operation").show(5, truncate=False)
except Exception as e:
    print("CDF append demo:", type(e).__name__, e)

Append OK — new history version should exist
+-------+-----------------+
|version|operation        |
+-------+-----------------+
|2      |WRITE            |
|1      |SET TBLPROPERTIES|
|0      |WRITE            |
+-------+-----------------+



In [0]:
# Read change feed from version 0 (API varies by DBR)
try:
    ch = spark.read.format("delta").option("readChangeFeed", "true").option("startingVersion", 0).load(P_CDF)
    print("Change feed schema:")
    ch.printSchema()
    ch.show(15, truncate=False)
except Exception as e:
    print("readChangeFeed:", type(e).__name__, str(e)[:220])
    print("→ Fallback: use DESCRIBE HISTORY + streaming module on Day 6")

Change feed schema:
root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: integer (nullable = true)
 |-- _change_type: string (nullable = true)
 |-- _commit_version: long (nullable = true)
 |-- _commit_timestamp: timestamp (nullable = true)

readChangeFeed: AnalysisException [DELTA_MISSING_CHANGE_DATA] Error getting change data for range [0 , 2] as change data was not
recorded for version [0]. If you've enabled change data feed on this table,
use `DESCRIBE HISTORY` to see when it was first e
→ Fallback: use DESCRIBE HISTORY + streaming module on Day 6


In [0]:
# SQL: table_changes (Databricks SQL — if available)
try:
    spark.sql(f"SELECT * FROM table_changes(delta.`{P_CDF}`, 0) LIMIT 20").show(truncate=False)
except Exception as e:
    print("table_changes SQL:", type(e).__name__, str(e)[:200])

table_changes SQL: AnalysisException [UNRESOLVED_COLUMN.WITHOUT_SUGGESTION] A column, variable, or function parameter with name `delta`.`abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/day5/delta/cdf_lab_demo` cannot be reso


## Section 7 — **CHECK** / **NOT NULL** constraints (Delta)

Delta supports **table constraints** on newer runtimes:

```sql
-- ALTER TABLE delta.`path` ADD CONSTRAINT cnt_nonneg CHECK (count >= 0);
```

Violations fail writes — great for Silver/Gold quality gates.

## Section 7b — Add a **CHECK** constraint (scratch table; try/except)

In [0]:
P_CHK = DAY5 + "/delta/check_constraint_demo"
spark.read.format("delta").load(P_BASIC).limit(100).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P_CHK)
try:
    spark.sql(f"ALTER TABLE delta.`{P_CHK}` ADD CONSTRAINT cnt_nonneg CHECK (count >= 0)")
    print("Constraint added (or already exists)")
except Exception as e:
    print("ADD CONSTRAINT:", type(e).__name__, str(e)[:200])

Constraint added (or already exists)


In [0]:
# Violation should fail the write
from pyspark.sql import Row

try:
    bad = spark.createDataFrame([Row(DEST_COUNTRY_NAME="X", ORIGIN_COUNTRY_NAME="Y", count=-1)])
    bad.write.format("delta").mode("append").save(P_CHK)
    print("Unexpected: negative row accepted")
except Exception as e:
    print("Expected failure on CHECK:", type(e).__name__, str(e)[:160])

Expected failure on CHECK: AnalysisException [DELTA_CONSTRAINT_DATA_TYPE_MISMATCH] Column count has data type INT and cannot be altered to data type BIGINT because this column is referenced by the followin


## Section 8 — Repeated **history** after a small write

In [0]:
from pyspark.sql.functions import lit

spark.read.format("delta").load(P_BASIC).limit(1).withColumn("audit_note", lit("history_demo")).write.format("delta").mode("append").option("mergeSchema", "true").save(P_BASIC)
spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`").select("version", "operation").show(5, truncate=False)

+-------+---------+
|version|operation|
+-------+---------+
|3      |WRITE    |
|2      |OPTIMIZE |
|1      |WRITE    |
|0      |WRITE    |
+-------+---------+



## Section 9 — Practice prompts

### P9.1 — Compare **row count** at **version 0** vs **latest** on `P_BASIC`. When do they differ?

### P9.2 — Run **OPTIMIZE** on `P_BASIC` and re-check **`DESCRIBE DETAIL`** (file counts may change).

### P9.3 — Explain why **time travel** protects you from bad **overwrite** during testing.

### P9.4 — Use **`DeltaTable.forPath`** + **`.history()`** to print **`operationMetrics`** for the last **MERGE** or **WRITE** you performed (from notebook **04** if available).

### P9.5 — Sketch a **Job** that runs **`OPTIMIZE`** then **`ANALYZE TABLE`** (if SQL warehouse) for a curated Gold table.

### P9.6 — What is the **default** retention for **`VACUUM`** and why is **7 days** the common safety margin?

## Section 10 — **Worked examples** (history + detail drills)

In [0]:
# Drill A — last 3 operations with userMetadata if set
spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`").select("version", "operation", "operationMetrics").show(3, truncate=False)

+-------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|operation|operationMetrics                                                                                                                                                                                                                    |
+-------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|3      |WRITE    |{numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 2511}                                                                                                                                                                         |
|2  

In [0]:
# Drill B — partition columns + minReaderVersion / minWriterVersion (if columns exist in DETAIL output)
d = spark.sql(f"DESCRIBE DETAIL delta.`{P_PART}`")
d.show(truncate=False)

+------+------------------------------------+----+-----------+------------------------------------------------------------------------------------------+----------------------+-------------------+----------------+-----------------+--------+-----------+-------------------------------------+----------------+----------------+-----------------------------------------+---------------------------------------------------------------+-------------+
|format|id                                  |name|description|location                                                                                  |createdAt             |lastModified       |partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties                           |minReaderVersion|minWriterVersion|tableFeatures                            |statistics                                                     |clusterByAuto|
+------+------------------------------------+----+-----------+------------------------------------------------

In [0]:
# Drill C — read same path with SQL vs DataFrame reader (row count parity)
sql_c = spark.sql(f"SELECT COUNT(*) c FROM delta.`{P_BASIC}`").collect()[0]["c"]
df_c = spark.read.format("delta").load(P_BASIC).count()
print(sql_c, df_c)

511 511


## Section 10d — More **history** / **detail** drills (run all)

In [0]:
# Drill D — operations that touched P_PART in last 8 commits
spark.sql(f"DESCRIBE HISTORY delta.`{P_PART}`").select("version", "timestamp", "operation").show(8, truncate=False)

+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|1      |2026-03-21 13:54:55|WRITE    |
|0      |2026-03-21 13:53:56|WRITE    |
+-------+-------------------+---------+



In [0]:
# Drill E — min/max Spark reader versions (if present in DETAIL)
d = spark.sql(f"DESCRIBE DETAIL delta.`{P_BASIC}`")
cols = [c for c in d.columns if "version" in c.lower() or "format" in c.lower()]
if cols:
    d.select(*cols).show(truncate=False)
else:
    d.show(3, truncate=False)

+------+----------------+----------------+
|format|minReaderVersion|minWriterVersion|
+------+----------------+----------------+
|delta |3               |7               |
+------+----------------+----------------+



In [0]:
# Drill F — time travel: full schema at version 0 vs latest
h = spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`")
v0 = h.selectExpr("min(version) as v").collect()[0]["v"]
v1 = h.selectExpr("max(version) as v").collect()[0]["v"]
print("versions", v0, v1)
if v0 is not None:
    spark.read.format("delta").option("versionAsOf", int(v0)).load(P_BASIC).printSchema()
print("--- latest ---")
spark.read.format("delta").load(P_BASIC).printSchema()

versions 0 3
root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: integer (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_path: string (nullable = true)

--- latest ---
root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: integer (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_path: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- load_comment: string (nullable = true)
 |-- audit_note: string (nullable = true)



In [0]:
# Drill G — ZORDER then re-check partition stats (P_PART)
try:
    spark.sql(f"OPTIMIZE delta.`{P_PART}` ZORDER BY (DEST_COUNTRY_NAME)")
except Exception as e:
    print("ZORDER:", type(e).__name__)
spark.sql(f"DESCRIBE DETAIL delta.`{P_PART}`").select("numFiles", "sizeInBytes").show(truncate=False)

+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|22      |35302      |
+--------+-----------+



## Section 11 — Recap quiz

Short answers (discuss in class or keep in your own notes):

1. Does time travel copy all data files each time, or mainly change which files the log points to?
2. Why does RESTORE add a new version instead of deleting old history?
3. Give one risk of running VACUUM with very short retention.
4. Does OPTIMIZE change logical query results, or mainly file layout?
5. When is ZORDER most useful?
6. Which table property enables Change Data Feed?
7. What happens to a write that violates a CHECK constraint?

---

## Section 12 — Liquid clustering / predictive optimization (brief note)

Newer runtimes may offer liquid clustering or automated maintenance. Check your Databricks runtime release notes for what applies in your workspace.

---

## End of notebook 03

Continue with `04-Day5-Delta-DML-MERGE-SCD.ipynb` for MERGE, UPDATE, DELETE, and SCD patterns.